# Lasana — Improved Forest Semantic Segmentation

**Task:** Binary forest / non-forest segmentation from satellite images  
**Upgrade from:** `Notebooks/Kalana/main.py` (simple CNN baseline)  
**Model:** U-Net + BatchNorm + skip connections (TensorFlow / Keras)

### What this notebook improves

| Area | Kalana | Lasana |
|------|--------|--------|
| Architecture | Plain encoder–decoder | **U-Net with skip connections** |
| Normalization | None | **BatchNorm** |
| Loss | BCE only | **BCE + Dice** |
| Metrics | Pixel accuracy | **IoU, Dice, Precision, Recall** (+ accuracy) |
| Mask resize | Default interpolation | **Nearest-neighbor** |
| Mask values | `/255` | **Threshold at 127 → 0/1** |
| Augmentation | None | **Flips, rotations, color jitter** |
| Split | 80/20 | **80/10/10 train/val/test** |
| Training | 10 epochs, fixed LR | **Up to 50 epochs, early stop, LR schedule** |

Read `README.md` in this folder for full explanations of **why** each choice is used.

---
### How to run
1. Point `IMAGE_FOLDER` / `MASK_FOLDER` to your dataset (cell below).
2. Runtime with GPU recommended.
3. Run all cells in order.

## 1 · Imports & reproducibility

**Why:** Fixing the random seed makes train/val/test splits and weight init comparable across runs so you can trust that gains come from the model, not luck.

In [ ]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    CSVLogger,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2 · Configuration & paths

**Why these settings**
- `IMG_SIZE=256` — matches the Forest Segmented patches and Kalana/Dinura baselines.
- `MASK_THRESHOLD=127` — JPEG masks are rarely pure 0/255; mid-gray pixels must be forced to 0 or 1.
- `80/10/10` split — validation drives early stopping / LR; test stays untouched for final report.
- Longer training + early stopping — U-Net needs more epochs than Kalana’s 10, but we stop when val IoU stalls.

In [ ]:
# ── Paths (edit these) ───────────────────────────────────────────────────────
# Option A: dataset next to Lasana under DNN_Project/data/...
# Option B: local images/ and masks/ folders inside Lasana/

BASE_DIR = os.path.abspath(os.getcwd())

# Try common locations in order; first existing pair wins.
_CANDIDATES = [
    (
        os.path.join(BASE_DIR, "dataset", "Forest Segmented", "Forest Segmented", "images"),
        os.path.join(BASE_DIR, "dataset", "Forest Segmented", "Forest Segmented", "masks"),
    ),
    (
        os.path.join(BASE_DIR, "images"),
        os.path.join(BASE_DIR, "masks"),
    ),
    (
        os.path.join(BASE_DIR, "..", "data", "Forest Segmented", "Forest Segmented", "images"),
        os.path.join(BASE_DIR, "..", "data", "Forest Segmented", "Forest Segmented", "masks"),
    ),
]

IMAGE_FOLDER = None
MASK_FOLDER = None
for img_c, mask_c in _CANDIDATES:
    if os.path.isdir(img_c) and os.path.isdir(mask_c):
        IMAGE_FOLDER, MASK_FOLDER = img_c, mask_c
        break

# Manual override if auto-detect fails:
# IMAGE_FOLDER = r"C:\path\to\images"
# MASK_FOLDER  = r"C:\path\to\masks"

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────────
IMG_SIZE = 256
MASK_THRESHOLD = 127
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10   # of full data; test = remainder (~0.10)

BATCH_SIZE = 8     # raise to 16 if GPU memory allows
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
BCE_WEIGHT = 0.5
DICE_WEIGHT = 0.5
EARLY_STOP_PATIENCE = 10
LR_PATIENCE = 5

BEST_CKPT = os.path.join(CHECKPOINT_DIR, "lasana_unet_best.keras")
LAST_CKPT = os.path.join(CHECKPOINT_DIR, "lasana_unet_last.keras")
LOG_CSV = os.path.join(RESULTS_DIR, "training_log.csv")

print("IMAGE_FOLDER:", IMAGE_FOLDER)
print("MASK_FOLDER :", MASK_FOLDER)
assert IMAGE_FOLDER and MASK_FOLDER, (
    "Could not find images/masks. Set IMAGE_FOLDER and MASK_FOLDER manually."
)

## 3 · Load image–mask pairs

**What this does**
- Matches `*_sat_*` images to `*_mask_*` files (same convention as Kalana).
- Resizes **images** with bilinear interpolation (smooth RGB).
- Resizes **masks** with **nearest-neighbor** so class labels stay crisp (no gray edges).
- Binarizes masks with threshold 127.

**Why nearest-neighbor for masks:** bilinear/cubic resize mixes neighboring labels into values like 0.4 — those are not valid class IDs and blur the learning target.

In [ ]:
def find_mask_path(mask_folder, filename):
    """Return best matching mask path for an image filename."""
    stem, ext = os.path.splitext(filename)
    common_exts = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]
    candidate_stems = [stem]
    if "_sat_" in stem:
        candidate_stems.append(stem.replace("_sat_", "_mask_"))

    ext_order = [ext] + [e for e in common_exts if e != ext]
    for candidate_stem in candidate_stems:
        for candidate_ext in ext_order:
            candidate = os.path.join(mask_folder, candidate_stem + candidate_ext)
            if os.path.exists(candidate):
                return candidate
    return None


def load_dataset(image_folder, mask_folder, img_size=256, mask_threshold=127):
    image_files = sorted(
        f for f in os.listdir(image_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"))
    )

    images, masks = [], []
    skipped = 0

    for file in image_files:
        image_path = os.path.join(image_folder, file)
        mask_path = find_mask_path(mask_folder, file)

        image = cv2.imread(image_path)
        if image is None:
            skipped += 1
            continue
        if mask_path is None:
            skipped += 1
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            skipped += 1
            continue

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
        image = image.astype(np.float32) / 255.0

        # CRITICAL: nearest-neighbor keeps hard 0/1 labels
        mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = (mask > mask_threshold).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    images = np.array(images, dtype=np.float32)
    masks = np.array(masks, dtype=np.float32)

    if len(images) == 0:
        raise RuntimeError("No valid image-mask pairs loaded. Check paths.")

    print(f"Loaded {len(images)} pairs | skipped {skipped}")
    print("Images:", images.shape, "| Masks:", masks.shape)
    print(f"Forest pixel ratio: {masks.mean():.4f}")
    return images, masks


images, masks = load_dataset(IMAGE_FOLDER, MASK_FOLDER, IMG_SIZE, MASK_THRESHOLD)

## 4 · Train / Val / Test split

**Why 80/10/10 instead of Kalana’s 80/20**
- **Train** — learn weights.
- **Val** — pick best checkpoint & tune LR / early stop (never used for final claim).
- **Test** — report once at the end so you do not overfit to the number you publish.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    images, masks, test_size=0.10, random_state=SEED
)
# Of remaining 90%: need ~10% of full data as val → 10/90 ≈ 0.111...
val_fraction_of_temp = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_fraction_of_temp, random_state=SEED
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# Free full arrays if memory is tight (optional)
del images, masks, X_temp, y_temp

## 5 · Data augmentation (train only)

**What:** random horizontal/vertical flips, 90° rotations, mild brightness/contrast changes.  
**Why:** satellite scenes appear at many orientations and lighting conditions; augmentation reduces overfitting without new labels.  
**Important:** val/test are **not** augmented — metrics must reflect real data.

In [ ]:
def augment_pair(image, mask):
    """Apply the same geometric transform to image and mask."""
    # Horizontal flip
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    # Vertical flip
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    # 0 / 90 / 180 / 270 rotation (k * 90°)
    k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    # Photometric jitter on image only (mask labels must stay fixed)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.clip_by_value(image, 0.0, 1.0)

    # Keep mask binary after geometric ops
    mask = tf.cast(mask > 0.5, tf.float32)
    return image, mask


def make_dataset(x, y, batch_size, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(x), 1024), seed=SEED)
    if augment:
        ds = ds.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(X_train, y_train, BATCH_SIZE, shuffle=True, augment=True)
val_ds = make_dataset(X_val, y_val, BATCH_SIZE, shuffle=False, augment=False)
test_ds = make_dataset(X_test, y_test, BATCH_SIZE, shuffle=False, augment=False)

print("Datasets ready.")

## 6 · U-Net model

**What each part does**
- **DoubleConv (Conv → BN → ReLU × 2):** extracts features; BatchNorm stabilizes activations.
- **Encoder + MaxPool:** grows channels, shrinks spatial size (context).
- **Bottleneck:** deepest abstract representation.
- **Decoder + UpSampling:** rebuilds resolution.
- **Skip connections (`concatenate`):** inject encoder detail into decoder so forest edges stay sharp.
- **1×1 Conv + sigmoid:** one probability per pixel (forest vs background).

**Why U-Net beats Kalana’s plain CNN:** without skips, fine boundaries are lost in the bottleneck and the decoder cannot recover them well.

In [ ]:
def conv_block(x, filters):
    """Two 3x3 convolutions with BatchNorm + ReLU."""
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x


def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), features=(64, 128, 256, 512)):
    inputs = Input(shape=input_shape)
    x = inputs
    skips = []

    # Encoder
    for f in features:
        x = conv_block(x, f)
        skips.append(x)
        x = layers.MaxPooling2D(2)(x)

    # Bottleneck
    x = conv_block(x, features[-1] * 2)

    # Decoder
    for f, skip in zip(reversed(features), reversed(skips)):
        x = layers.UpSampling2D(2)(x)
        x = layers.Concatenate()([skip, x])
        x = conv_block(x, f)

    outputs = layers.Conv2D(1, 1, activation="sigmoid", padding="same")(x)
    return Model(inputs, outputs, name="lasana_unet")


model = build_unet()
model.summary()

## 7 · Loss & metrics

**BCE** — good per-pixel probability calibration.  
**Dice loss** — `1 − Dice`; directly rewards overlap of predicted forest with ground truth.  
**Combined loss** — balances both (same idea as Dinura’s `BCEDiceLoss`).

**IoU / Dice metrics** — what you should report; pixel accuracy alone can look high while forest regions are wrong.

In [ ]:
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    bce = tf.reduce_mean(bce)
    return BCE_WEIGHT * bce + DICE_WEIGHT * dice_loss(y_true, y_pred)


def iou_coef(y_true, y_pred, threshold=0.5, smooth=1.0):
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred_bin, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=bce_dice_loss,
    metrics=[
        "accuracy",
        dice_coef,
        iou_coef,
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

print("Compiled with BCE + Dice loss; tracking accuracy, Dice, IoU, Precision, Recall.")

## 8 · Train

**Callbacks**
- **ModelCheckpoint (best val IoU)** — keep the best forest-overlap model, not the last epoch.
- **EarlyStopping** — stop when val IoU stops improving (saves time, fights overfitting).
- **ReduceLROnPlateau** — lower LR when learning stalls.
- **CSVLogger** — save history for plots / reports.

In [ ]:
callbacks = [
    ModelCheckpoint(
        BEST_CKPT,
        monitor="val_iou_coef",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    ModelCheckpoint(
        LAST_CKPT,
        save_best_only=False,
        verbose=0,
    ),
    EarlyStopping(
        monitor="val_iou_coef",
        mode="max",
        patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_iou_coef",
        mode="max",
        factor=0.5,
        patience=LR_PATIENCE,
        min_lr=1e-6,
        verbose=1,
    ),
    CSVLogger(LOG_CSV),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS,
    callbacks=callbacks,
)

print(f"Best checkpoint: {BEST_CKPT}")

## 9 · Training curves

**Why plot IoU/Dice, not only loss:** loss can drop while overlap quality stalls. Curves show whether the model is still learning useful forest masks.

In [ ]:
hist = history.history
epochs_range = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_range, hist["loss"], label="train")
axes[0].plot(epochs_range, hist["val_loss"], label="val")
axes[0].set_title("BCE + Dice Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs_range, hist["iou_coef"], label="train")
axes[1].plot(epochs_range, hist["val_iou_coef"], label="val")
axes[1].set_title("IoU")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(epochs_range, hist["dice_coef"], label="train")
axes[2].plot(epochs_range, hist["val_dice_coef"], label="val")
axes[2].set_title("Dice")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
curve_path = os.path.join(RESULTS_DIR, "training_curves.png")
plt.savefig(curve_path, dpi=150)
plt.show()
print("Saved:", curve_path)

## 10 · Test-set evaluation

Load the **best val-IoU** weights and evaluate once on the held-out test set.  
Compare **IoU / Dice** to Kalana’s pixel accuracy — a fairer view of forest segmentation quality.

In [ ]:
# Reload best weights (EarlyStopping already restored them, but this is explicit)
if os.path.exists(BEST_CKPT):
    model.load_weights(BEST_CKPT)
    print("Loaded best checkpoint.")

results = model.evaluate(test_ds, return_dict=True)
print("\n=== Test metrics ===")
for k, v in results.items():
    print(f"{k:12s}: {v:.4f}")

## 11 · Visual predictions

Side-by-side: original image | ground-truth mask | prediction.  
**What to look for:** sharper forest boundaries than Kalana, fewer missed patches, less “all background” collapse.

In [ ]:
n_show = min(4, len(X_test))
preds = model.predict(X_test[:n_show], verbose=0)
preds_bin = (preds > 0.5).astype(np.float32)

fig, axes = plt.subplots(n_show, 3, figsize=(10, 3 * n_show))
if n_show == 1:
    axes = np.expand_dims(axes, 0)

for i in range(n_show):
    axes[i, 0].imshow(X_test[i])
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(y_test[i].squeeze(), cmap="gray")
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(preds_bin[i].squeeze(), cmap="gray")
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
pred_path = os.path.join(RESULTS_DIR, "prediction_grid.png")
plt.savefig(pred_path, dpi=150)
plt.show()
print("Saved:", pred_path)

## 12 · Summary

You trained an improved baseline that addresses Kalana’s main weaknesses:

1. **U-Net + skips + BatchNorm** → better spatial detail  
2. **BCE + Dice** → optimizes forest overlap, not only pixel-wise BCE  
3. **IoU / Dice metrics** → honest segmentation scores  
4. **Nearest-neighbor masks + threshold 127** → clean labels  
5. **Augmentation + early stopping + LR schedule** → better generalization  

**Artifacts**
- `checkpoints/lasana_unet_best.keras`
- `results/training_log.csv`
- `results/training_curves.png`
- `results/prediction_grid.png`

**Next ideas:** pretrained encoder (ResNet/EfficientNet), Focal loss, threshold tuning on val IoU, or compare against Dinura’s PyTorch U-Net on the same split.